In [1]:
import pyodbc
import pandas as pd

cnxn = pyodbc.connect('DSN=Hermes_DSN',autocommit=True)
cursor = cnxn.cursor()

In [2]:
isin = pd.read_csv('key dataframe\\bond_day.csv')

In [4]:
isin_unique = tuple(isin['isin'].unique())

In [ ]:
query = f"""

WITH sector_volumes AS (
  -- Total Lending by mapped EA-Sector
  SELECT 
    reference_period,
    (CASE WHEN lender_country IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') 
          THEN 'EA' ELSE 'non-EA' END) || ' - ' || 
    (CASE lender_sector 
          WHEN 'S122' THEN 'BANK'
          WHEN 'S124' THEN 'IF'
          WHEN 'S125' THEN 'OFI'
          WHEN 'S128' THEN 'ICPF'
          WHEN 'S129' THEN 'ICPF'
          ELSE 'Other' END) AS ea_sector,
    SUM(loan_value_eur_clean) AS lending_vol,
    0 AS borrowing_vol
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE loan_price_currency = 'USD'
  AND isin IN {isin_unique}
  AND LEFT(isin, 2) = 'US'
  GROUP BY reference_period, ea_sector

  UNION ALL

  -- Total Borrowing by mapped EA-Sector
  SELECT 
    reference_period,
    (CASE WHEN borrower_country IN ('AT','BE','HR','CY','EE','FI','FR','DE','GR','IE','IT','LV','LT','LU','MT','NL','PT','SK','SI','ES') 
          THEN 'EA' ELSE 'non-EA' END) || ' - ' || 
    (CASE borrower_sector 
          WHEN 'S122' THEN 'BANK'
          WHEN 'S124' THEN 'IF'
          WHEN 'S125' THEN 'OFI'
          WHEN 'S128' THEN 'ICPF'
          WHEN 'S129' THEN 'ICPF'
          ELSE 'Other' END) AS ea_sector,
    0 AS lending_vol,
    SUM(loan_value_eur_clean) AS borrowing_vol
  FROM xlab_ecb_prj_sftds_cb_common.hermesf_sl
  WHERE loan_price_currency = 'USD'
  AND isin IN {isin_unique}
    AND LEFT(isin, 2) = 'US'
  GROUP BY reference_period, ea_sector
)

SELECT 
  reference_period,
  ea_sector,
  SUM(lending_vol) AS total_lending,
  SUM(borrowing_vol) AS total_borrowing,
  SUM(lending_vol) - SUM(borrowing_vol) AS net_position
FROM sector_volumes
GROUP BY reference_period, ea_sector
ORDER BY reference_period, ea_sector;

"""
df_us = pd.read_sql_query(query, cnxn)

C:\Users\hermesf\AppData\Local\Temp\ipykernel_19484\4087958159.py:58: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df_us = pd.read_sql_query(query, cnxn)


In [8]:
df_us

,reference_period,ea_sector,total_lending,total_borrowing,net_position
0,2021-01-01,EA - BANK,8.562377e+11,8.313239e+10,7.731053e+11
1,2021-01-01,EA - ICPF,4.135543e+08,4.204814e+08,-6.927015e+06
2,2021-01-01,EA - IF,3.146819e+09,4.882786e+09,-1.735967e+09
3,2021-01-01,EA - OFI,3.599409e+10,3.887034e+09,3.210705e+10
4,2021-01-01,EA - Other,2.375494e+08,3.351486e+05,2.372142e+08
...,...,...,...,...,...
14686,2026-09-16,EA - OFI,0.000000e+00,1.616968e+09,-1.616968e+09
14687,2026-09-16,non-EA - BANK,0.000000e+00,8.929580e+09,-8.929580e+09
14688,2026-09-16,non-EA - IF,4.807450e+06,0.000000e+00,4.807450e+06
14689,2026-09-16,non-EA - OFI,1.090004e+07,4.268710e+09,-4.257810e+09
